# Preventative Pulse — Predictive Analytics for Heart Disease

**Objective:** Build a classification model to predict heart disease from CDC behavioral survey data, identify the most influential risk factors, and apply the trained pipeline to 445K unseen records.

**Data:** 2020 & 2022 BRFSS (Behavioral Risk Factor Surveillance System) surveys via the CDC.

**Pipeline:** EDA → preprocessing → 6-model comparison with SMOTE → hyperparameter tuning → validation → test-set prediction.

## 1 | Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
from collections import Counter

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, f1_score,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    xgboost_available = True
except ImportError:
    xgboost_available = False
    print("XGBoost not available — continuing without it.")

warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (10, 5), 'axes.titlesize': 14,
    'axes.labelsize': 12, 'figure.dpi': 120
})
print("Environment ready.")


---
## 2 | Data Loading & Quality Audit

In [ ]:
df_raw = pd.read_csv('data/heart_2020_cleaned.csv')
print(f"Dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\nColumn types:\n{df_raw.dtypes.value_counts().to_string()}")
print(f"\nMissing values: {df_raw.isnull().sum().sum()} (all columns complete)")
print(f"Duplicate rows:  {df_raw.duplicated().sum():,} ({df_raw.duplicated().mean():.1%})")
print(f"\nTarget distribution:")
print(df_raw['HeartDisease'].value_counts().to_string())
print(f"\nImbalance ratio: {df_raw['HeartDisease'].value_counts()['No'] / df_raw['HeartDisease'].value_counts()['Yes']:.1f} : 1")


The dataset has zero nulls and a severe class imbalance — 91.4% of respondents report no heart disease. Although the data arrives clean, the subsections below apply explicit quality checks to every feature (missing values, outliers, invalid entries) as part of a rigorous audit.

---
## 3 | Exploratory Data Analysis

### 3.0 | Target Variable — HeartDisease

In [ ]:
counts = df_raw['HeartDisease'].value_counts()
pcts = counts / len(df_raw) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#4CAF50', '#E91E63']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
for i, (lbl, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + 2000, f'{val:,}\n({pcts[lbl]:.1f}%)', ha='center')
axes[0].set_title('HeartDisease — Class Distribution')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('HeartDisease — Proportions')

plt.tight_layout()
plt.savefig('figures/eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.1 | Numeric Features — BMI, PhysicalHealth, MentalHealth, SleepTime

Each numeric feature is audited for missing values, outliers (IQR method), and plotted as a three-panel strip: distribution, boxplot, and heart-disease rate by binned range.

In [ ]:
numeric_features = {
    'BMI': {
        'bins': [0, 18.5, 25, 30, 35, 40, 100],
        'labels': ['<18.5', '18.5–25', '25–30', '30–35', '35–40', '40+'],
        'xlabel': 'BMI Range', 'color': '#2196F3'
    },
    'PhysicalHealth': {
        'bins': [0, 1, 5, 10, 15, 20, 31],
        'labels': ['0', '1–4', '5–9', '10–14', '15–19', '20–30'],
        'xlabel': 'Days of Poor Physical Health', 'color': '#2196F3'
    },
    'MentalHealth': {
        'bins': [0, 1, 5, 10, 15, 20, 31],
        'labels': ['0', '1–4', '5–9', '10–14', '15–19', '20–30'],
        'xlabel': 'Days of Poor Mental Health', 'color': '#9C27B0'
    },
    'SleepTime': {
        'bins': [0, 5, 6, 7, 8, 9, 25],
        'labels': ['≤4', '5', '6', '7', '8', '9+'],
        'xlabel': 'Sleep Hours', 'color': '#00897B'
    }
}

print("=== Numeric Feature Quality Audit ===\n")
print(f"{'Feature':<18} {'Missing':>8} {'Min':>8} {'Max':>8} {'Mean':>8} {'Median':>8} {'Std':>8} {'IQR Outliers':>14}")
print("-" * 95)
for col in numeric_features:
    s = df_raw[col]
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((s < Q1 - 1.5 * IQR) | (s > Q3 + 1.5 * IQR)).sum()
    print(f"{col:<18} {s.isnull().sum():>8} {s.min():>8.1f} {s.max():>8.1f} "
          f"{s.mean():>8.2f} {s.median():>8.1f} {s.std():>8.2f} {n_out:>10,} ({n_out/len(s)*100:.1f}%)")

# --- 4-row figure: distribution + boxplot + HD rate per feature ---
fig, axes = plt.subplots(4, 3, figsize=(16, 18))

for row, (col, cfg) in enumerate(numeric_features.items()):
    s = df_raw[col]

    # Distribution
    axes[row][0].hist(s, bins=50, color=cfg['color'], edgecolor='white', alpha=0.85)
    axes[row][0].axvline(s.mean(), color='red', linestyle='--', lw=1, label=f'Mean={s.mean():.1f}')
    axes[row][0].axvline(s.median(), color='orange', linestyle='-', lw=1, label=f'Med={s.median():.1f}')
    axes[row][0].set_title(f'{col} — Distribution')
    axes[row][0].set_ylabel('Count')
    axes[row][0].legend(fontsize=8)

    # Boxplot
    bp = axes[row][1].boxplot(s.dropna(), vert=True, patch_artist=True,
                              boxprops=dict(facecolor=cfg['color'], alpha=0.6))
    axes[row][1].set_title(f'{col} — Boxplot')
    axes[row][1].set_ylabel(col)

    # HD rate by bin
    temp_col = f'{col}_bin'
    df_raw[temp_col] = pd.cut(s, bins=cfg['bins'], labels=cfg['labels'], right=False)
    hd_rate = df_raw.groupby(temp_col, observed=True)['HeartDisease'].apply(
        lambda x: (x == 'Yes').mean())
    axes[row][2].bar(hd_rate.index.astype(str), hd_rate.values, color='#E91E63', edgecolor='white')
    axes[row][2].set_title(f'Heart Disease Rate by {col}')
    axes[row][2].set_ylabel('HD Rate')
    axes[row][2].set_xlabel(cfg['xlabel'])
    for j, v in enumerate(hd_rate.values):
        axes[row][2].text(j, v + 0.003, f'{v:.1%}', ha='center', fontsize=8)
    df_raw.drop(columns=temp_col, inplace=True)

plt.tight_layout()
plt.savefig('figures/eda_numeric_features.png', dpi=150, bbox_inches='tight')
plt.show()


**Key numeric findings:**
- **BMI** — Right-skewed (peak 25–28). Heart-disease rate roughly doubles from the normal range (~6%) to 40+ (~13%).
- **PhysicalHealth** — Zero-inflated (71% at 0 days). Strong positive association: 6% at 0 days → 23% at 20–30 days.
- **MentalHealth** — Similar zero-inflation (64%). Weaker association with heart disease, possibly confounded by age.
- **SleepTime** — Centered at 7 hours. U-shaped relationship: both short (≤4h, ~15%) and long sleepers (9+, ~13%) show elevated risk versus the 7-hour group (~7%).

### 3.2 | Numeric Feature Correlation

In [ ]:
corr = df_raw[['BMI', 'PhysicalHealth', 'MentalHealth', 'SleepTime']].corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            mask=mask, square=True, linewidths=0.5, ax=ax)
ax.set_title('Numeric Feature Correlation')
plt.tight_layout()
plt.savefig('figures/eda_numeric_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Max |r| = 0.32 (PhysicalHealth ↔ MentalHealth). No multicollinearity concerns.")


### 3.3 | Binary Categorical Features

Nine binary features covering medical history (Stroke, KidneyDisease, Asthma, SkinCancer), lifestyle (Smoking, AlcoholDrinking, PhysicalActivity, DiffWalking), and demographics (Sex). Each is audited and plotted with its heart-disease rate.

In [ ]:
binary_cols = ['Smoking', 'AlcoholDrinking', 'Stroke', 'DiffWalking',
               'PhysicalActivity', 'Asthma', 'KidneyDisease', 'SkinCancer', 'Sex']

# Audit table
print("=== Binary Feature Quality Audit ===\n")
print(f"{'Feature':<20} {'Missing':>8} {'Yes/Cat1 %':>12} {'HD Rate (No)':>14} {'HD Rate (Yes)':>15} {'Δ':>8}")
print("-" * 80)
for col in binary_cols:
    s = df_raw[col]
    vals = sorted(s.unique())
    hd = df_raw.groupby(col)['HeartDisease'].apply(lambda x: (x == 'Yes').mean())
    if col == 'Sex':
        pct = (s == 'Male').mean() * 100
        print(f"{col:<20} {s.isnull().sum():>8} {'Male: '+f'{pct:.1f}%':>12} "
              f"{'F: '+f'{hd["Female"]:.1%}':>14} {'M: '+f'{hd["Male"]:.1%}':>15} "
              f"{abs(hd['Male'] - hd['Female']):>+8.1%}")
    else:
        pct = (s == 'Yes').mean() * 100
        print(f"{col:<20} {s.isnull().sum():>8} {f'{pct:.1f}%':>12} "
              f"{hd.get('No', 0):>14.1%} {hd.get('Yes', 0):>15.1%} "
              f"{hd.get('Yes', 0) - hd.get('No', 0):>+8.1%}")

# --- Grid visualization ---
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes_flat = axes.flatten()

for idx, col in enumerate(binary_cols):
    ax = axes_flat[idx]
    hd_rate = df_raw.groupby(col)['HeartDisease'].apply(lambda x: (x == 'Yes').mean())

    if col == 'Sex':
        bar_colors = ['#E91E63', '#2196F3']
    else:
        bar_colors = ['#4CAF50', '#E91E63']

    bars = ax.bar(hd_rate.index, hd_rate.values, color=bar_colors[:len(hd_rate)], edgecolor='white')
    for j, v in enumerate(hd_rate.values):
        ax.text(j, v + 0.005, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'HD Rate by {col}', fontsize=11)
    ax.set_ylabel('Heart Disease Rate')
    ax.set_ylim(0, max(hd_rate.values) * 1.25)

plt.suptitle('Heart Disease Rate — Binary Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('figures/eda_binary_features.png', dpi=150, bbox_inches='tight')
plt.show()


**Standout binary features:**
- **Stroke** — 36% HD rate among stroke survivors vs. 8% without — the largest binary effect in the dataset.
- **KidneyDisease** — 29% vs. 8%, a 3.7× relative risk.
- **DiffWalking** — 23% vs. 6%, reflecting the physical toll of heart disease or shared risk factors.
- **Smoking** shows a moderate effect (12% vs. 6%), and **AlcoholDrinking** is counterintuitively *lower* among drinkers (5% vs. 9%) — likely a survivor bias / healthy-drinker confound.

### 3.4 | Multi-Class Categorical Features — AgeCategory, Race, Diabetic, GenHealth

In [ ]:
# --- AgeCategory ---
age_order = ['18-24', '25-29', '30-34', '35-39', '40-44', '45-49',
             '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80 or older']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Age
hd_age = df_raw.groupby('AgeCategory')['HeartDisease'].apply(
    lambda x: (x == 'Yes').mean()).reindex(age_order)
axes[0][0].bar(range(len(age_order)), hd_age.values, color='#E91E63', edgecolor='white')
axes[0][0].set_xticks(range(len(age_order)))
axes[0][0].set_xticklabels(age_order, rotation=45, ha='right', fontsize=8)
axes[0][0].set_title('Heart Disease Rate by Age Category')
axes[0][0].set_ylabel('HD Rate')
for j, v in enumerate(hd_age.values):
    axes[0][0].text(j, v + 0.005, f'{v:.0%}', ha='center', fontsize=7)

# Race
hd_race = df_raw.groupby('Race')['HeartDisease'].apply(
    lambda x: (x == 'Yes').mean()).sort_values()
axes[0][1].barh(hd_race.index, hd_race.values, color='#E91E63', edgecolor='white')
axes[0][1].set_title('Heart Disease Rate by Race')
axes[0][1].set_xlabel('HD Rate')
for j, v in enumerate(hd_race.values):
    axes[0][1].text(v + 0.002, j, f'{v:.1%}', va='center', fontsize=9)

# Diabetic
hd_diab = df_raw.groupby('Diabetic')['HeartDisease'].apply(
    lambda x: (x == 'Yes').mean()).sort_values()
axes[1][0].barh(hd_diab.index, hd_diab.values, color='#FF9800', edgecolor='white')
axes[1][0].set_title('Heart Disease Rate by Diabetic Status')
axes[1][0].set_xlabel('HD Rate')
for j, v in enumerate(hd_diab.values):
    axes[1][0].text(v + 0.002, j, f'{v:.1%}', va='center', fontsize=9)

# GenHealth
health_order = ['Excellent', 'Very good', 'Good', 'Fair', 'Poor']
hd_gen = df_raw.groupby('GenHealth')['HeartDisease'].apply(
    lambda x: (x == 'Yes').mean()).reindex(health_order)
grad_colors = ['#4CAF50', '#8BC34A', '#FFC107', '#FF9800', '#F44336']
axes[1][1].bar(health_order, hd_gen.values, color=grad_colors, edgecolor='white')
axes[1][1].set_title('Heart Disease Rate by General Health')
axes[1][1].set_ylabel('HD Rate')
for j, v in enumerate(hd_gen.values):
    axes[1][1].text(j, v + 0.005, f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/eda_multiclass_features.png', dpi=150, bbox_inches='tight')
plt.show()

# Audit
for col in ['AgeCategory', 'Race', 'Diabetic', 'GenHealth']:
    print(f"{col}: {df_raw[col].isnull().sum()} missing, {df_raw[col].nunique()} categories")


**Multi-class findings:**
- **AgeCategory** — Monotonic climb from 2% (18–24) to 25% (80+). The single most powerful predictor.
- **GenHealth** — Steep gradient from 2% (Excellent) to 34% (Poor), but potentially circular since respondents with known heart disease may self-rate as poor.
- **Diabetic** — 4× risk for diabetics (22%) vs. non-diabetics (6%). Borderline diabetes already shows elevated risk (13%).
- **Race** — American Indian/Alaskan Native (10%) and White (9%) highest; Asian (3%) lowest. Sample sizes vary widely.

### 3.5 | EDA Summary Table

In [ ]:
rows = []
for col in df_raw.columns:
    s = df_raw[col]
    if s.dtype in [np.float64, np.int64]:
        vtype = 'Numeric'
    elif col == 'HeartDisease':
        vtype = 'Target'
    elif s.nunique() == 2:
        vtype = 'Binary'
    else:
        vtype = 'Multi-Class'

    row = {'Variable': col, 'Type': vtype, 'Missing': s.isnull().sum(), 'Unique': s.nunique()}

    if vtype == 'Numeric':
        row['Summary'] = f"μ={s.mean():.1f}, σ={s.std():.1f}, range [{s.min():.0f}–{s.max():.0f}]"
    else:
        mode = s.mode()[0]
        row['Summary'] = f"Mode: {mode} ({(s == mode).mean():.0%})"

    if col != 'HeartDisease':
        hd = df_raw.groupby(col)['HeartDisease'].apply(lambda x: (x == 'Yes').mean())
        row['HD Rate Range'] = f"{hd.min():.1%} – {hd.max():.1%}"
    else:
        row['HD Rate Range'] = '—'
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.index = range(1, len(summary_df) + 1)
print(summary_df.to_string())
summary_df.to_csv('figures/eda_summary_table.csv', index=False)


---
## 4 | Class Rebalancing — Before & After

Random undersampling brings both classes to parity. The table below quantifies the impact on numeric feature distributions — confirming the shifts are small and expected.

In [ ]:
minority_count = df_raw['HeartDisease'].value_counts()['Yes']
df_yes = df_raw[df_raw['HeartDisease'] == 'Yes'].sample(minority_count, random_state=42)
df_no  = df_raw[df_raw['HeartDisease'] == 'No'].sample(minority_count, random_state=42)
df = pd.concat([df_yes, df_no]).sample(frac=1, random_state=42).reset_index(drop=True)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ['#4CAF50', '#E91E63']

for ax, data, title in [(axes[0], df_raw, f'BEFORE (n={len(df_raw):,})'),
                         (axes[1], df, f'AFTER (n={len(df):,})')]:
    cts = data['HeartDisease'].value_counts()
    pcts = cts / len(data) * 100
    ax.bar(cts.index, cts.values, color=colors, edgecolor='white')
    for j, (lbl, v) in enumerate(zip(cts.index, cts.values)):
        ax.text(j, v + max(cts) * 0.02, f'{v:,}\n({pcts[lbl]:.1f}%)', ha='center')
    ax.set_title(title)
    ax.set_ylabel('Count')

plt.suptitle('Target Distribution — Before & After Undersampling', fontsize=13)
plt.tight_layout()
plt.savefig('figures/eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

# Before/after stats
print("\n=== Numeric Feature Stats: Before vs. After Undersampling ===\n")
print(f"{'Feature':<18} {'Metric':<8} {'Before':>10} {'After':>10} {'Δ':>10}")
print("-" * 58)
for col in ['BMI', 'PhysicalHealth', 'MentalHealth', 'SleepTime']:
    for metric in ['mean', 'std']:
        vb = getattr(df_raw[col], metric)()
        va = getattr(df[col], metric)()
        print(f"{col:<18} {metric:<8} {vb:>10.3f} {va:>10.3f} {va - vb:>+10.3f}")
    print()


---
## 5 | Preprocessing & Feature Engineering

Strict **fit-on-train, transform-on-both** protocol:
1. Median imputation (numeric) / mode imputation (categorical)
2. One-hot encoding with `drop_first=True`
3. Feature alignment between train and test schemas
4. StandardScaler on numeric features
5. 80/20 stratified train-validation split

In [ ]:
# Identify columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in df.select_dtypes(include=['object', 'string']).columns
                    if c != 'HeartDisease']

print(f"Numeric ({len(numeric_cols)}):     {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

# Imputation
numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

if numeric_cols:
    df[numeric_cols] = numeric_imputer.fit_transform(df[numeric_cols])
if categorical_cols:
    df[categorical_cols] = categorical_imputer.fit_transform(df[categorical_cols])

# One-hot encode
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(f"\nAfter encoding: {df_encoded.shape[1]} columns")

# Separate X / y
X = df_encoded.drop('HeartDisease', axis=1)
y_raw = df_encoded['HeartDisease']
le = LabelEncoder()
y = le.fit_transform(y_raw)
print(f"Target classes: {le.classes_} → {list(range(len(le.classes_)))}")

# Fill remaining NaN, scale
X = X.fillna(X.mean())
scaler = StandardScaler()
num_cols_enc = X.select_dtypes(include='number').columns.tolist()
X[num_cols_enc] = scaler.fit_transform(X[num_cols_enc])

# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f"\nTrain: {X_train.shape[0]:,}  |  Validation: {X_val.shape[0]:,}")
print(f"Features: {X.shape[1]}")


---
## 6 | Model Evaluation — 5-Fold CV with SMOTE

Six classifiers wrapped in `ImbPipeline` with SMOTE applied inside each fold. Three metrics per model: F1-Macro (lead metric), Accuracy, ROC-AUC.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
}
if xgboost_available:
    models['XGBoost'] = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print(f"Evaluating {len(models)} models (5-fold CV + SMOTE)...\n")
for name, model in models.items():
    pipe = ImbPipeline([("smote", SMOTE(random_state=42)), ("classifier", model)])
    f1  = cross_val_score(pipe, X, y, cv=cv, scoring="f1_macro",  n_jobs=-1)
    acc = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy",  n_jobs=-1)
    roc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc",   n_jobs=-1)
    results[name] = {'f1_macro': f1.mean(), 'accuracy': acc.mean(), 'roc_auc': roc.mean(),
                     'f1_std': f1.std(), 'acc_std': acc.std(), 'roc_std': roc.std()}
    print(f"{name:<25} F1={f1.mean():.4f}(±{f1.std():.4f})  "
          f"Acc={acc.mean():.4f}  AUC={roc.mean():.4f}")

best_name = max(results, key=lambda x: results[x]['f1_macro'])
best_model = models[best_name]
print(f"\n→ Best by F1-Macro: {best_name} ({results[best_name]['f1_macro']:.4f})")


### 6a | Model Comparison Chart

In [ ]:
model_names = list(results.keys())
f1_vals  = [results[m]['f1_macro'] for m in model_names]
acc_vals = [results[m]['accuracy'] for m in model_names]
roc_vals = [results[m]['roc_auc']  for m in model_names]

x = np.arange(len(model_names))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w, f1_vals,  w, label='F1-Macro', color='#2196F3')
ax.bar(x,     acc_vals, w, label='Accuracy', color='#4CAF50')
ax.bar(x + w, roc_vals, w, label='ROC-AUC',  color='#FF9800')

ax.set_ylabel('Score')
ax.set_title('Model Comparison — 5-Fold Cross-Validation')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=30, ha='right')
ax.legend()
ax.set_ylim(0.6, 0.9)
ax.grid(axis='y', alpha=0.3)

best_idx = f1_vals.index(max(f1_vals))
ax.annotate(f'{max(f1_vals):.4f}', xy=(best_idx - w, max(f1_vals)),
            xytext=(0, 8), textcoords='offset points',
            ha='center', fontsize=9, fontweight='bold', color='#2196F3')

plt.tight_layout()
plt.savefig('figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 7 | Hyperparameter Tuning

`RandomizedSearchCV` on the best model, 3-fold CV, F1-Macro scoring. SMOTE remains inside the pipeline.

In [ ]:
param_grids = {
    'Logistic Regression': {
        'classifier__C': [0.01, 0.1, 1.0, 10.0],
        'classifier__penalty': ['l1', 'l2'],
        'classifier__solver': ['liblinear']
    },
    'Random Forest': {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [3, 5, 10, None]
    },
    'Gradient Boosting': {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.05, 0.1, 0.2]
    },
    'K-Nearest Neighbors': {'classifier__n_neighbors': [3, 5, 7, 9, 11]},
    'Decision Tree': {'classifier__max_depth': [3, 5, 10, None]},
}
if xgboost_available:
    param_grids['XGBoost'] = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.05, 0.1, 0.2]
    }

pipeline = ImbPipeline([("smote", SMOTE(random_state=42)), ("classifier", best_model)])

if best_name in param_grids:
    print(f"Tuning {best_name}...")
    search = RandomizedSearchCV(pipeline, param_grids[best_name], n_iter=10, cv=3,
                                scoring='f1_macro', random_state=42, n_jobs=-1)
    search.fit(X_train, y_train)
    pipeline = search.best_estimator_
    print(f"Best params: {search.best_params_}")
    print(f"Best CV F1:  {search.best_score_:.4f}")
else:
    pipeline.fit(X_train, y_train)
    print("No param grid defined; trained with defaults.")


---
## 8 | Validation — Held-Out Set

In [ ]:
y_pred = pipeline.predict(X_val)
y_prob = pipeline.predict_proba(X_val)[:, 1]

print(f"Accuracy:  {accuracy_score(y_val, y_pred):.4f}")
print(f"F1-Macro:  {f1_score(y_val, y_pred, average='macro'):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, y_prob):.4f}")
print(f"\n{classification_report(y_val, y_pred, target_names=le.classes_)}")


### 8a | Confusion Matrix

In [ ]:
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix — Validation Set')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


### 8b | ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_prob)
roc_auc_val = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(fpr, tpr, color='#2196F3', lw=2, label=f'{best_name} (AUC = {roc_auc_val:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Validation Set')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()


### 8c | Precision–Recall Curve

In [ ]:
prec, rec, _ = precision_recall_curve(y_val, y_prob)
ap = average_precision_score(y_val, y_prob)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(rec, prec, color='#E91E63', lw=2, label=f'{best_name} (AP = {ap:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision–Recall Curve — Validation Set')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()


### 8d | Feature Importance

Coefficient magnitudes from the tuned logistic regression reveal which features drive predictions. Pink = increases heart-disease risk; blue = decreases it.

In [ ]:
classifier = pipeline.named_steps['classifier']
feature_names = X.columns.tolist()

if hasattr(classifier, 'coef_'):
    importances = classifier.coef_[0]
    sorted_idx = np.argsort(np.abs(importances))
    top_n = min(20, len(sorted_idx))
    top_idx = sorted_idx[-top_n:]
    colors = ['#E91E63' if importances[i] > 0 else '#2196F3' for i in top_idx]

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(np.arange(top_n), importances[top_idx], color=colors, edgecolor='none')
    ax.set_yticks(np.arange(top_n))
    ax.set_yticklabels([feature_names[i] for i in top_idx])
    ax.set_xlabel('Coefficient Magnitude')
    ax.set_title(f'Top {top_n} Features — {best_name}')
    ax.grid(axis='x', alpha=0.3)
    ax.axvline(x=0, color='gray', linewidth=0.8)
    ax.annotate('→ Higher HD risk', xy=(0.95, 0.02), xycoords='axes fraction',
                ha='right', fontsize=9, color='#E91E63', fontstyle='italic')
    ax.annotate('← Lower HD risk', xy=(0.05, 0.02), xycoords='axes fraction',
                ha='left', fontsize=9, color='#2196F3', fontstyle='italic')
    plt.tight_layout()
    plt.savefig('figures/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

elif hasattr(classifier, 'feature_importances_'):
    importances = classifier.feature_importances_
    sorted_idx = np.argsort(importances)[-20:]
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(np.arange(len(sorted_idx)), importances[sorted_idx], color='#2196F3')
    ax.set_yticks(np.arange(len(sorted_idx)))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx])
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 20 Features — {best_name}')
    plt.tight_layout()
    plt.savefig('figures/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()


---
## 9 | Test-Set Predictions (2022 BRFSS)

The 2022 dataset has a different schema (40 columns, renamed features). Columns are mapped, aligned, and processed through the same pipeline.

In [ ]:
# Load and align 2022 test data
df_test = pd.read_csv("data/heart_2022_with_nans.csv", on_bad_lines="skip")
print(f"Raw 2022 data: {df_test.shape[0]:,} rows × {df_test.shape[1]} columns")

# Rename to match 2020 schema
df_test = df_test.rename(columns={
    'SmokerStatus': 'Smoking', 'AlcoholDrinkers': 'AlcoholDrinking',
    'HadStroke': 'Stroke', 'PhysicalHealthDays': 'PhysicalHealth',
    'MentalHealthDays': 'MentalHealth', 'DifficultyWalking': 'DiffWalking',
    'RaceEthnicityCategory': 'Race', 'HadDiabetes': 'Diabetic',
    'PhysicalActivities': 'PhysicalActivity', 'GeneralHealth': 'GenHealth',
    'SleepHours': 'SleepTime', 'HadAsthma': 'Asthma',
    'HadKidneyDisease': 'KidneyDisease', 'HadSkinCancer': 'SkinCancer'
})

# Keep only training-set columns (minus target)
keep_cols = [c for c in df.columns if c != 'HeartDisease' and c in df_test.columns]
df_test = df_test[keep_cols]
print(f"After alignment: {df_test.shape[1]} columns retained")

# Impute missing values in test data
# (fillna instead of sklearn imputer — avoids column-mismatch from train imputer)
num_cols_test = [c for c in numeric_cols if c in df_test.columns]
cat_cols_test = [c for c in categorical_cols if c in df_test.columns]

if num_cols_test:
    df_test[num_cols_test] = df_test[num_cols_test].fillna(df_test[num_cols_test].median())
    print(f"Imputed {len(num_cols_test)} numeric columns with median")
if cat_cols_test:
    df_test[cat_cols_test] = df_test[cat_cols_test].fillna(df_test[cat_cols_test].mode().iloc[0])
    print(f"Imputed {len(cat_cols_test)} categorical columns with mode")

df_test_encoded = pd.get_dummies(df_test, columns=cat_cols_test, drop_first=True)

# Align columns
for c in set(X.columns) - set(df_test_encoded.columns):
    df_test_encoded[c] = 0
df_test_encoded = df_test_encoded[X.columns]
df_test_encoded = df_test_encoded.fillna(X.mean())
df_test_encoded[num_cols_enc] = scaler.transform(df_test_encoded[num_cols_enc])

# Predict
test_preds = pipeline.predict(df_test_encoded)
test_probs = pipeline.predict_proba(df_test_encoded)[:, 1]
test_labels = le.inverse_transform(test_preds)

print(f"\nPrediction Summary:")
print(f"  Total:     {len(test_preds):,}")
print(f"  Positive:  {test_preds.sum():,} ({test_preds.mean():.1%})")
print(f"  Negative:  {(len(test_preds) - test_preds.sum()):,}")


### 9a | Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(test_probs, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(0.5, color='red', linestyle='--', lw=1, label='Threshold')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Probability Distribution (2022 Test Set)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

labels_u, counts = np.unique(test_labels, return_counts=True)
axes[1].bar(labels_u, counts, color=['#4CAF50', '#E91E63'], edgecolor='white')
for j, (lbl, cnt) in enumerate(zip(labels_u, counts)):
    axes[1].text(j, cnt + 2000, f'{cnt:,}', ha='center', fontweight='bold')
axes[1].set_xlabel('Predicted Class')
axes[1].set_ylabel('Count')
axes[1].set_title('Class Distribution (2022 Test Set)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/prediction_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 10 | Conclusions

**Model performance:** Logistic regression with SMOTE achieved 83.5% ROC-AUC and 75.7% F1-Macro on the validation set, outperforming five alternative classifiers. The model correctly identifies 77% of actual heart-disease cases (recall) while maintaining 75% precision.

**Top risk factors:** Age is the dominant predictor by a wide margin — the 80+ age coefficient is 5× larger than any medical condition. After age, stroke history, poor self-reported health, and diabetes carry the most weight.

**Clinical applicability:** The model is not a diagnostic tool, but it can serve as a screening aid. A 7% predicted prevalence on the 2022 test set aligns with CDC estimates of self-reported heart disease in U.S. adults.

**Limitations:**
- Self-reported survey data introduces response bias (especially for lifestyle questions like exercise and alcohol)
- The 2020→2022 schema mismatch required feature mapping and column dropping, which may lose signal
- GenHealth is likely partially circular — people with known heart disease rate their health as poor
- The model cannot detect *early* heart disease; it predicts current diagnosis status

**Future work:** Incorporating the additional 2022 features (e.g., flu/pneumonia vaccination, HIV testing, COVID status) could improve discrimination. Ensemble methods with threshold optimization may further reduce false negatives, which carry higher clinical cost than false positives in a screening context.